# Package

In [6]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

# Importation des données

In [7]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    # IMPORTANT: full_feature_names=True pour éviter collisions / noms stables
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence via calendrier mensuel + filtrage implicite par Feast
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

# Option robuste: on "valide" les dates réellement présentes en prenant UNRATE
entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

# normaliser date
df_stationary["date"] = pd.to_datetime(df_stationary["date"], utc=True, errors="coerce").dt.tz_convert(None)

# colonne valeur stable
value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = df_stationary[["series_id", "date", value_col]].rename(columns={value_col: "value"})
print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Construire dataset régression (ciblé sur UNRATE)
#    y = UNRATE
#    y_lag12 = lag 12 de UNRATE
#    exog = autres séries contemporaines
# ----------------------------
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)
df_y["y_lag12"] = df_y["y"].shift(12)

df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x_long = df_x_long[df_x_long["date"].isin(df_y["date"])]

# exog en colonnes (uniquement sur dates UNRATE)
df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .reset_index()
)

# merge final
df_model = (
    df_y[["date", "y", "y_lag12"]]
    .merge(df_x, on="date", how="left")
    .dropna()
)

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
ts_lr shape: (776, 14)
Exog cols: ['y_lag12', 'BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


,unique_id,ds,y,y_lag12,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
12,UNRATE,1961-01-01,1.4,-0.8,0.009124,-0.003725,-0.013323,-0.032283,-0.002848,0.0,0.000025,0.105695,-0.06,1.0
13,UNRATE,1961-02-01,2.1,-1.1,-0.002222,-0.003712,-0.006225,-0.019280,0.004903,0.0,0.008219,0.114030,0.05,1.0
14,UNRATE,1961-03-01,1.5,-0.2,-0.018140,-0.005726,0.018488,0.006067,0.005823,0.0,0.016896,0.121220,0.14,0.0
15,UNRATE,1961-04-01,1.8,0.0,-0.008034,-0.004027,0.016472,0.025197,0.003544,0.0,0.010125,0.097409,0.05,0.0
16,UNRATE,1961-05-01,2.0,0.0,-0.002025,-0.002013,0.018928,0.041699,-0.000003,0.0,0.013953,0.067329,-0.13,0.0


# Dictionnaire de modèle

In [ ]:
MLF_MODELS = {
    "LR_LAG12_MANUAL_EXOG": lambda freq: MLForecast(
        models=[LinearRegression()],
        freq=freq,
        lags=[], 
    )
}

# Backtesting

In [18]:
from mlforecast.utils import PredictionIntervals

def run_backtesting_h12_simple(
    mlf,
    ts,
    *,
    h=12,
    step_size=12,
    partitions=4,
    pi_windows=3,
    levels=[95],
):
    pi = PredictionIntervals(
        h=h,
        n_windows=pi_windows,
        method="conformal_distribution",
    )

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
        static_features=[],   # ✅ FIX: toutes les features sont dynamiques
    )

    return bkt_df

# Run 

In [19]:
# ============================================================
# RUN – Linear Regression (Nixtla MLForecast)
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

H = 12
STEP_SIZE = 12
PARTITIONS = 35
PI_WINDOWS = 3
LEVELS = [95]

# instantiate model
mlf = MLF_MODELS["LR_LAG12_AUTO"](FREQ)

# ✅ safe prints for MLForecast
model_names = list(mlf.models.keys())
first_name = next(iter(mlf.models))
print("Running models:", model_names)
print("First model class:", mlf.models[first_name].__class__.__name__)
print("Freq:", FREQ)

# backtesting unchanged
bkt_lr = run_backtesting_h12_simple(
    mlf=mlf,
    ts=ts_lr,
    h=H,
    step_size=STEP_SIZE,
    partitions=PARTITIONS,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
)

bkt_lr.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA
Running models: ['LinearRegression']
First model class: LinearRegression
Freq: MS


,unique_id,ds,cutoff,y,LinearRegression,LinearRegression-lo-95,LinearRegression-hi-95
0,UNRATE,1990-09-01,1990-08-01,0.6,0.639219,-0.266885,1.545322
1,UNRATE,1990-10-01,1990-08-01,0.6,0.763641,0.316744,1.210538
2,UNRATE,1990-11-01,1990-08-01,0.8,1.419879,0.459885,2.379873
3,UNRATE,1990-12-01,1990-08-01,0.9,1.766807,0.599155,2.934459
4,UNRATE,1991-01-01,1990-08-01,1.0,1.768316,1.648329,1.888303


In [21]:
# ============================================================
# (ADD) Build standardized OOS forecast table (Linear Regression)
# ============================================================

# Identifier automatiquement les colonnes modèle
model_col = [c for c in bkt_lr.columns if c.lower() == "linearregression"][0]
lo_col = [c for c in bkt_lr.columns if c.lower() == "linearregression-lo-95"][0]
hi_col = [c for c in bkt_lr.columns if c.lower() == "linearregression-hi-95"][0]

df_lr_forecasts = (
    bkt_lr[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat_lr",
        lo_col: "y_hat_lr_lo_95",
        hi_col: "y_hat_lr_hi_95",
    })
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

print("df_lr_forecasts shape:", df_lr_forecasts.shape)
df_lr_forecasts.head()

df_lr_forecasts shape: (420, 7)


,series_id,date,cutoff,y_obs,y_hat_lr,y_hat_lr_lo_95,y_hat_lr_hi_95
0,UNRATE,1990-09-01,1990-08-01,0.6,0.639219,-0.266885,1.545322
1,UNRATE,1990-10-01,1990-08-01,0.6,0.763641,0.316744,1.210538
2,UNRATE,1990-11-01,1990-08-01,0.8,1.419879,0.459885,2.379873
3,UNRATE,1990-12-01,1990-08-01,0.9,1.766807,0.599155,2.934459
4,UNRATE,1991-01-01,1990-08-01,1.0,1.768316,1.648329,1.888303


In [22]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (Linear Regression)
# ============================================================

from datetime import datetime
import json

# ----------------------------
# Model identification
# ----------------------------
SERIES_ID = "UNRATE"
MODEL_TAG = "lr_lag12_exog"   # 🔑 clair et extensible

# ----------------------------
# Directories
# ----------------------------
OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) Outputs (OOS forecasts)
# ----------------------------
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_oos_forecasts.parquet"
df_lr_forecasts.to_parquet(oos_path, index=False)

# ----------------------------
# 2) Artifacts – raw backtesting output
# ----------------------------
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_bkt_raw.parquet"
bkt_lr.to_parquet(bkt_path, index=False)

# ----------------------------
# 3) Run configuration (reproducibility)
# ----------------------------
run_config = {
    "model": "LinearRegression",
    "framework": "Nixtla-MLForecast",
    "target": SERIES_ID,
    "stationary": True,
    "lags": [12],
    "exogenous_variables": [
        c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]
    ],
    "horizon": H,
    "step_size": STEP_SIZE,
    "partitions": PARTITIONS,
    "prediction_intervals": {
        "method": "conformal_distribution",
        "levels": LEVELS,
        "n_windows": PI_WINDOWS,
    },
    "frequency": FREQ,
}

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ----------------------------
# 4) Metadata – run info
# ----------------------------
meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "model_tag": MODEL_TAG,
    "files": {
        "oos_forecasts": str(oos_path.resolve()),
        "cv_raw": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved:")
print(" - OOS forecasts :", oos_path.name)
print(" - CV raw        :", bkt_path.name)
print(" - Config        :", cfg_path.name)
print(" - Metadata      :", meta_path.name)


✅ Saved:
 - OOS forecasts : unrate_lr_lag12_exog_oos_forecasts.parquet
 - CV raw        : unrate_lr_lag12_exog_bkt_raw.parquet
 - Config        : unrate_lr_lag12_exog_config.json
 - Metadata      : unrate_lr_lag12_exog_run_info.json


C:\Users\Mita\AppData\Local\Temp\ipykernel_5780\677556810.py:72: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_utc": datetime.utcnow().isoformat(),


# Graphique

In [26]:
# ----------------------------
# Prepare obs
# ----------------------------
df_obs = (
    df_lr_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

In [27]:
# ----------------------------
# Prepare forecast + PI (LR)
# ----------------------------
df_fcst = (
    df_lr_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_lr": "LR",
        "y_hat_lr_lo_95": "LR-lo-95",
        "y_hat_lr_hi_95": "LR-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "LR",
        "LR-lo-95",
        "LR-hi-95",
    ]]
)

In [28]:
from utilsforecast.plotting import plot_series

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Rename legend entries
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (stationary)"
    elif trace.name == "LR":
        trace.name = "Linear Regression (lag-12 + exog)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 